# CTD Profile Grapher

Interactive depth profiles from Sea-Bird `.cnv` files — no Excel step, no Google Drive.

**How to use**
1. Run **Setup** once per session.
2. Run **1 · Survey location and files** — type where you sampled, pick your `.cnv` files, then tick which variables you want graphed. The ones the notebook recognises are already ticked; anything else your files contain is listed too, in case you want it. Depth is the Y axis unless you change it.
3. Run **2 · Draw the graphs**. To look at part of the water column, fill in `depth_from_m` and `depth_to_m` and run that cell again. Your files stay loaded, so there is no need to upload twice.

**Naming:** the filename becomes the legend label, with underscores turned into spaces — `Station_1.cnv` → **Station 1**, `East_Passage.cnv` → **East Passage**. Stations sort naturally (1, 2, … 10, 11).

**Colours** are locked to station order, so the first station is always the same blue, the second always the same red, and so on. The Load cell prints the colour key so a bar chart or a station map can use exactly the same colours.

## What you get

Files land in `/content/CTD_output/` — open the folder icon in the left sidebar to download them.

- **`png/*.png`** — ordinary pictures, one per variable. Use these anywhere you would use a photo.
- **`CTD_profiles.html`** — all the graphs together, interactive, around 30 KB. Hovering shows exact values; you can zoom, or hide a station by clicking it in the legend. Needs internet to open.
- **`single_graphs/*.html`** — the same graphs but **one file each**: `Temperature.html`, `Salinity.html`, and so on. Take just the one you want.
- **`CTD_profiles_offline.html`** — all the graphs with everything built in, several MB, works with no connection at all. Only worth taking if you will be presenting somewhere without wifi.

## Sharing a graph

**For a document or slideshow — use the PNG.** Google Docs, Word, Google Slides and PowerPoint cannot display an interactive chart, and no setting or add-on changes that. Insert the picture like any other image.

**To let someone explore it — send them `CTD_profiles.html`.** They double-click it and it opens in their browser with everything working, nothing to install. This is the easy answer and it covers almost every case.

**Want just one graph, not all of them?** Use the matching file from `single_graphs/` — `Temperature.html` is the depth vs temperature graph on its own, and nothing else. Treat it exactly like the file above: send it, or put it online for a link. Each one is about 15 KB.

**To put a graph inside a web page**, hand the HTML file to whoever manages the site and ask them to embed it in an iframe. That is a normal request and they will know what it means. Google Drive will not work as a substitute — it downloads the file rather than displaying it.

The sensor set is read from each file's own header, so casts from different instruments work with no setting to change.

## Credit

Example data collected by students of **TGEOS 445, Estuarine Field Studies, University of Washington Tacoma**, Spring 2026, in Colvos Passage and East Passage, Puget Sound.

Instrument: **Sea-Bird SBE 19plus** (temperature and conductivity SN 7686), processed with Sea-Bird SBEDataProcessing.

In [ ]:
#@title Setup — run once per session
!pip install -q "kaleido==0.2.1" 2>/dev/null

import os, re, colorsys
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ─────────────── settings ───────────────
LINE_SHAPE   = "spline"              # "spline" (smooth) or "linear" (raw bins)
SHOW_MARKERS = False                 # True → dot at each 1-db bin
LINE_WIDTH   = 1.5
X_PAD_FRAC   = 0.05                  # breathing room at left/right edges (5%)
Y_PAD_FRAC   = 0.02
AXIS_FONT    = 13                    # x and y axis titles share this size
EXPORT_PNG   = True
OUT_DIR      = "/content/CTD_output"
LOCATION     = ""                    # set by the Load cell, e.g. "Quartermaster Harbor"
# ────────────────────────────────────────

# (label, ticked by default, [(short name, unit), ...])
# The classic set is ticked; derived and engineering channels are recognised so
# they get a proper name and unit, but stay unticked so the default output does
# not balloon. Names surveyed across 51 public .cnv files from SBE 9, 19plus,
# 21 and 25plus instruments.
VARIABLES = [
    ("Temperature", True,
     [("tv290c", "°C"), ("t090c", "°C"), ("t190c", "°C"), ("tv190c", "°C"),
      ("t068c", "°C"), ("t168c", "°C"), ("t090", "°C"), ("t190", "°C"),
      ("t4990c", "°C"), ("tnc90c", "°C")]),
    ("Potential Temperature", False,
     [("potemp090c", "°C"), ("potemp190c", "°C"), ("potemp068c", "°C"),
      ("potemp168c", "°C")]),
    ("Salinity", True, [("sal00", "PSU"), ("sal11", "PSU")]),
    ("Conductivity", False,
     [("c0s/m", "S/m"), ("c1s/m", "S/m"), ("c0ms/cm", "mS/cm"),
      ("c1ms/cm", "mS/cm"), ("c0us/cm", "µS/cm"), ("c1us/cm", "µS/cm"),
      ("cond0s/m", "S/m")]),
    ("Density (sigma-t)", True,
     [("sigma-t00", "kg/m³"), ("sigma-t11", "kg/m³"), ("sigma-e00", "kg/m³"),
      ("sigma-é00", "kg/m³"), ("sigma-é11", "kg/m³"), ("density00", "kg/m³"),
      ("density11", "kg/m³")]),
    ("Sound Velocity", False,
     [("svcm", "m/s"), ("avgsvcm", "m/s"), ("svcm1", "m/s")]),
    ("Dissolved Oxygen", True,
     [("sbeox0ml/l", "mL/L"), ("sbeox1ml/l", "mL/L"), ("oxml/l", "mL/L"),
      ("sbeox0mm/kg", "µmol/kg"), ("sbeox1mm/kg", "µmol/kg"),
      ("sbeox0mg/l", "mg/L"), ("sbeox1mg/l", "mg/L"),
      ("sbeox0ps", "% sat"), ("sbeox1ps", "% sat")]),
    ("Oxygen Solubility", False,
     [("oxsolmm/kg", "µmol/kg"), ("oxsolml/l", "mL/L"), ("oxsolmg/l", "mg/L")]),
    ("Oxygen Saturation", False,
     [("oxsatmm/kg", "µmol/kg"), ("oxsatml/l", "mL/L"), ("oxsatmg/l", "mg/L")]),
    ("Fluorescence", True,
     [("fleco-afl", "mg/m³"), ("flecoafl", "mg/m³"), ("flcuva", "mg/m³"),
      ("flsp", "mg/m³"), ("flc", "mg/m³"), ("wetstar", "mg/m³")]),
    ("Beam Transmission", True,
     [("cstartr0", "%"), ("cstartr1", "%"), ("xmiss", "%")]),
    ("Beam Attenuation", False,
     [("bat", "1/m"), ("cstarat0", "1/m"), ("cstarat1", "1/m")]),
    ("Turbidity", True,
     [("turbwetntu0", "NTU"), ("turbwetntu1", "NTU"), ("obs", "NTU"),
      ("obs3+", "NTU"), ("seaturbmtr", "NTU"), ("upoly0", "NTU")]),
    ("pH", True, [("ph", "")]),
    ("PAR", True, [("par", "µmol photons/m²/s"), ("spar", "µmol photons/m²/s")]),
    ("CDOM", True, [("wetcdom", "mg/m³")]),
    ("Specific Volume Anomaly", False, [("sva", "10⁻⁸ m³/kg")]),
    ("Thermosteric Anomaly", False, [("tsa", "10⁻⁸ m³/kg")]),
]

# Vertical axis. Depth is preferred; pressure stands in when a file has no depth
# channel, and is labelled as pressure rather than quietly called metres.
DEPTH_CANDIDATES = [("depsm", "Depth (m)"), ("depfm", "Depth (m)"),
                    ("depsf", "Depth (fathoms)"),
                    ("prdm", "Pressure (db)"), ("prsm", "Pressure (db)"),
                    ("prm", "Pressure (db)"), ("prde", "Pressure (db)"),
                    ("pr", "Pressure (db)"), ("prdb", "Pressure (db)")]

# Fixed station palette. Position decides colour: the 1st station in the
# canonical order is always PALETTE[0], the 2nd always PALETTE[1], and so on.
# Reuse STATION_COLORS in a bar chart or a station map and the colours agree
# across every figure you make.
PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd",
           "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]

STATION_COLORS = {}
STATION_META = {}          # station label -> parse_header_meta() result
_EXTRA_COLORS = []          # generated colours beyond PALETTE, in order
_LAB = {}


def _to_lab(hexcolor):
    """sRGB hex → CIE Lab, so colours can be compared the way an eye does.
    Plain RGB distance calls greens near-identical that clearly are not."""
    if hexcolor in _LAB:
        return _LAB[hexcolor]
    r, g, b = (int(hexcolor[i:i + 2], 16) / 255 for i in (1, 3, 5))
    inv = lambda c: c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = inv(r), inv(g), inv(b)
    x = (0.4124 * r + 0.3576 * g + 0.1805 * b) / 0.95047
    y = 0.2126 * r + 0.7152 * g + 0.0722 * b
    z = (0.0193 * r + 0.1192 * g + 0.9505 * b) / 1.08883
    f = lambda t: t ** (1 / 3) if t > 0.008856 else 7.787 * t + 16 / 116
    x, y, z = f(x), f(y), f(z)
    _LAB[hexcolor] = (116 * y - 16, 500 * (x - y), 200 * (y - z))
    return _LAB[hexcolor]


def _candidate_colors():
    out = []
    for h in range(36):
        for light in (0.36, 0.50, 0.64):
            for sat in (0.50, 0.75, 0.95):
                r, g, b = colorsys.hls_to_rgb(h / 36, light, sat)
                out.append("#{:02x}{:02x}{:02x}".format(
                    round(r * 255), round(g * 255), round(b * 255)))
    return out


def _ensure_colors(n):
    """Grow the colour list to n entries, each new colour chosen as the one
    furthest from every colour already in use. Greedy farthest-point picking
    keeps large sets readable where evenly-spaced hues do not."""
    if len(PALETTE) + len(_EXTRA_COLORS) >= n:
        return
    cands = _candidate_colors()
    used = [_to_lab(c) for c in PALETTE + _EXTRA_COLORS]
    while len(PALETTE) + len(_EXTRA_COLORS) < n:
        best, best_d = None, -1.0
        for c in cands:
            lc = _to_lab(c)
            d = min((lc[0] - u[0]) ** 2 + (lc[1] - u[1]) ** 2 + (lc[2] - u[2]) ** 2
                    for u in used)
            if d > best_d:
                best_d, best = d, c
        _EXTRA_COLORS.append(best)
        used.append(_to_lab(best))


def station_color(i):
    """Colour for the i-th station. Lines are always solid, so past the base
    palette new colours are generated rather than repeated. Deterministic:
    station i gets the same colour every run, for any n."""
    if i < len(PALETTE):
        return PALETTE[i]
    _ensure_colors(i + 1)
    return _EXTRA_COLORS[i - len(PALETTE)]


def assign_station_styles(labels):
    """Lock each station to a colour by its position in the canonical order.

    Call once after loading. The returned dict is the colour key for this
    survey — reuse it in any other chart of the same stations."""
    STATION_COLORS.clear()
    for i, lab in enumerate(labels):
        STATION_COLORS[lab] = station_color(i)
    return STATION_COLORS


def natkey(s):
    """Sort so Station 2 comes before Station 10."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


# Colab's uploader never overwrites: upload Station_1.cnv twice and the second
# copy lands as "Station_1 (1).cnv", counting up on each repeat.
DUP_SUFFIX = re.compile(r"\s*\((\d+)\)\s*$")


def station_label(filename):
    """Station_1.cnv -> 'Station 1';  East_Passage.cnv -> 'East Passage'.
    A trailing ' (1)' left by a repeated upload is dropped."""
    base = os.path.splitext(os.path.basename(filename))[0]
    return DUP_SUFFIX.sub("", base).replace("_", " ").strip()


def upload_generation(filename):
    """How many times this file has been re-uploaded. Colab counts upward, so
    the highest number is the most recent copy."""
    base = os.path.splitext(os.path.basename(filename))[0]
    m = DUP_SUFFIX.search(base)
    return int(m.group(1)) if m else 0


def dedupe_uploads(raw):
    """Keep only the newest copy of each station.

    Re-uploading to fix a mistake would otherwise plot the station twice — once
    from the bad file and once from the good one — and shift every station's
    colour, since colour follows position."""
    best = {}
    for fn in raw:
        lab, gen = station_label(fn), upload_generation(fn)
        if lab not in best or gen > best[lab][0]:
            best[lab] = (gen, fn)
    keep = {fn for _, fn in best.values()}
    for fn in raw:
        if fn not in keep:
            print(f"  IGNORED  {fn} — superseded by a newer upload of "
                  f"'{station_label(fn)}'")
    return {fn: raw[fn] for fn in raw if fn in keep}


def find_channel(df, pairs):
    """First matching (short name, unit) pair, case-insensitive.
    Returns (column, unit), or (None, None) if the file carries none of them.
    First occurrence wins when a name appears twice."""
    lower = {}
    for c in df.columns:
        lower.setdefault(str(c).lower(), c)
    for name, unit in pairs:
        if name in lower:
            return lower[name], unit
    return None, None


def downcast_only(df, depth_col):
    """Keep the downcast only: cut at the deepest reading, then keep readings
    deeper than everything before them. Returns (cleaned, rows_removed).

    This makes a profile readable. It is NOT Sea-Bird's processing and is no
    substitute for it — see the Raw casts section of the README, and
    https://github.com/HakaiInstitute/seabird-processing"""
    if depth_col is None or len(df) < 3:
        return df, 0
    d = pd.to_numeric(df[depth_col], errors="coerce").to_numpy(dtype=float)
    if np.all(np.isnan(d)):
        return df, 0
    down = df.iloc[:int(np.nanargmax(d)) + 1]
    dd = pd.to_numeric(down[depth_col], errors="coerce").to_numpy(dtype=float)
    running = np.maximum.accumulate(np.nan_to_num(dd, nan=-np.inf))
    cleaned = down[dd >= running]
    return cleaned, len(df) - len(cleaned)


# Instrument bookkeeping rather than measurements — never offered as plottable.
HOUSEKEEPING = {"scan", "flag", "nbin", "pumps", "bpos", "pla", "timej", "times",
                "timem", "timek", "timen", "latitude", "longitude", "dz/dtm", "accm",
                "altm", "nbf", "prfails", "moderror", "seconds"}


def _nmea_degrees(text):
    """'25 55.56 S' -> -25.926.  Sea-Bird writes degrees, decimal minutes and a
    hemisphere; south and west are negative."""
    m = re.match(r"\s*(\d+)\s+([\d.]+)\s*([NSEW])", text.strip(), re.I)
    if not m:
        return None
    deg = int(m.group(1)) + float(m.group(2)) / 60.0
    return -deg if m.group(3).upper() in ("S", "W") else deg


def parse_header_meta(text):
    """Pull cast metadata out of a .cnv header.

    Headers carry three tiers: '*' instrument config, '**' free-form discovery
    metadata (ship, station, water depth), and '#' processing and parameters.
    Coordinates live in the '*' NMEA lines and are present in most real files,
    so a transect rarely needs them typed in by hand.

    Every field is optional — returns None for anything absent."""
    head = text.split("*END*")[0]
    out = {"lat": None, "lon": None, "time": None, "station": None,
           "ship": None, "cruise": None, "water_depth": None, "instrument": None}

    for line in head.splitlines():
        s = line.strip()
        m = re.match(r"\*\s*NMEA\s+Latitude\s*=\s*(.+)", s, re.I)
        if m:
            out["lat"] = _nmea_degrees(m.group(1))
            continue
        m = re.match(r"\*\s*NMEA\s+Longitude\s*=\s*(.+)", s, re.I)
        if m:
            out["lon"] = _nmea_degrees(m.group(1))
            continue
        # NMEA time is often literally "none"; fall back to start_time.
        m = re.match(r"\*\s*NMEA\s+UTC\s*\(Time\)\s*=\s*(.+)", s, re.I)
        if m and "none" not in m.group(1).lower():
            out["time"] = out["time"] or m.group(1).strip()
            continue
        m = re.match(r"#\s*start_time\s*=\s*([^\[]+)", s, re.I)
        if m:
            out["time"] = out["time"] or m.group(1).strip()
            continue
        if out["instrument"] is None:
            m = re.match(r"\*\s*(Sea-?Bird\s+SBE[^\r\n]*?)\s*Data File", s, re.I)
            if m:
                out["instrument"] = m.group(1).strip()
                continue
        # '**' lines are free-form "Key: value" or "Key = value"; only a handful
        # of files use them, and the keys are not standardised.
        m = re.match(r"\*\*\s*([^:=]+)[:=]\s*(.+)", s)
        if m:
            key, val = m.group(1).strip().lower(), m.group(2).strip()
            if not val:
                continue
            if "stat" in key:
                out["station"] = out["station"] or val
            elif "ship" in key or "vessel" in key:
                out["ship"] = out["ship"] or val
            elif "cruise" in key:
                out["cruise"] = out["cruise"] or val
            elif "depth" in key:
                d = re.search(r"[-+]?\d*\.?\d+", val)
                if d:
                    out["water_depth"] = float(d.group())
    return out


def _col_by_name(df, name):
    """Exact column lookup, case-insensitive."""
    for c in df.columns:
        if str(c).lower() == str(name).lower():
            return c
    return None


def available_channels(stations):
    """What these casts can plot.

    Returns (recognised, extras, y_default). `recognised` are the variables the
    notebook knows by name, with units. `extras` is every other measured column
    the files happen to carry — offered so nothing is hidden, just unticked by
    default. `y_default` is the depth or pressure channel."""
    recognised, seen = [], set()
    for label, on_by_default, cands in VARIABLES:
        for _st, df in stations:
            col, unit = find_channel(df, cands)
            if col is not None:
                recognised.append({"name": label, "col": str(col), "on": on_by_default,
                                   "label": f"{label} ({unit})" if unit else label})
                seen.add(str(col).lower())
                break

    ycol, ylab = None, None
    for _st, df in stations:
        c, l = find_channel(df, DEPTH_CANDIDATES)
        if c is not None:
            ycol, ylab = str(c), l
            break
    if ycol:
        seen.add(ycol.lower())

    extras = []
    for _st, df in stations:
        for c in df.columns:
            lc = str(c).lower()
            if lc in seen or lc in HOUSEKEEPING or (lc.startswith("v") and lc[1:].isdigit()):
                continue
            seen.add(lc)
            extras.append({"name": str(c), "col": str(c), "label": str(c)})

    # is_depth drives the depth window and the surface anchoring; invert only
    # flips the axis. They are separate so unticking "invert" does not quietly
    # disable the depth window as well.
    y_default = {"name": (ylab or "Depth (m)").split(" (")[0],
                 "col": ycol, "label": ylab or "Depth (m)",
                 "is_depth": True, "invert": True}
    return recognised, extras, y_default


def parse_cnv(text, source_name):
    """Parse a Sea-Bird .cnv. Columns come from the '# name' header lines and the
    data starts after *END*, so header length never has to be hardcoded."""
    lines = text.splitlines()
    col_names, bad_flag, end_idx, processing = {}, -9.99e-29, None, set()

    for i, raw in enumerate(lines):
        s = raw.strip()
        if s.upper() == "*END*":
            end_idx = i
            break
        m = re.match(r"#\s*name\s+(\d+)\s*=\s*([^:]+):", s)
        if m:
            col_names[int(m.group(1))] = m.group(2).strip()
            continue
        m = re.match(r"#\s*bad_flag\s*=\s*(\S+)", s)
        if m:
            try:
                bad_flag = float(m.group(1))
            except ValueError:
                pass
            continue
        low = s.lower()
        for tag in ("loopedit", "binavg", "wfilter", "filter", "derive",
                    "alignctd", "celltm", "split", "wildedit"):
            if low.startswith("# " + tag):
                processing.add(tag)

    if end_idx is None:
        raise ValueError(f"{source_name}: no *END* marker — is this a Sea-Bird .cnv?")
    if not col_names:
        raise ValueError(f"{source_name}: no '# name' column definitions in header.")

    ordered = [col_names[k] for k in sorted(col_names)]
    rows = []
    for raw in lines[end_idx + 1:]:
        parts = raw.split()
        if len(parts) != len(ordered):
            continue
        try:
            rows.append([float(x) for x in parts])
        except ValueError:
            continue

    df = pd.DataFrame(rows, columns=ordered)
    if not df.empty:
        # bad_flag is ~1e-29, so the comparison must be purely relative (atol=0),
        # otherwise every near-zero reading would be wiped out.
        mask = np.isclose(df.values.astype(float), bad_flag, rtol=1e-6, atol=0.0)
        df = df.mask(pd.DataFrame(mask, index=df.index, columns=df.columns))
    return df, processing


def load_files():
    """Show the upload button and read whatever is picked."""
    from google.colab import files as colab_files
    print("Select one or more .cnv files:")
    # .cnv headers are cp1252, not UTF-8 (e.g. the theta in sigma-theta)
    return dedupe_uploads(
        {n: b.decode("latin-1") for n, b in colab_files.upload().items()})


def _padded(lo, hi, frac):
    """Range with breathing room. Falls back sensibly if the series is flat."""
    span = hi - lo
    pad = span * frac if span > 0 else (abs(hi) * frac if hi else 1.0) or 1.0
    return lo - pad, hi + pad


def build_figures(stations, series, y_channel, depth_min=None, depth_max=None):
    """One figure per entry in `series`, every station overlaid.

    series      [{"name","label","col"}, ...]  x axis, one figure each
    y_channel   {"name","label","col","invert"}  shared y axis

    y_channel["is_depth"] says whether the depth window and surface anchoring
    apply; y_channel["invert"] only flips the axis direction, and is the user's
    tick box. Put a non-depth variable on y and it becomes an ordinary scatter —
    salinity against temperature, say — where no depth window makes sense."""
    if not STATION_COLORS:
        assign_station_styles([s for s, _ in stations])
    is_depth = bool(y_channel.get("is_depth", False))
    invert = bool(y_channel.get("invert", False))
    ylabel = y_channel["label"]
    figs = []

    for s in series:
        traces = []
        xlo = ylo = np.inf
        xhi = yhi = -np.inf
        for i, (st, df) in enumerate(stations):
            xcol = _col_by_name(df, s["col"])
            ycol = _col_by_name(df, y_channel["col"])
            if xcol is None or ycol is None:
                continue
            # The depth window filters readings by how deep they were taken, so it
            # applies whatever is on the axes — including salinity against
            # temperature, where depth is not plotted at all.
            dcol, _ = find_channel(df, DEPTH_CANDIDATES)
            cols = list(dict.fromkeys([c for c in (xcol, ycol, dcol) if c is not None]))
            sub = df[cols].dropna(subset=[xcol, ycol])
            if dcol is not None and depth_min is not None:
                sub = sub[sub[dcol] >= depth_min]
            if dcol is not None and depth_max is not None:
                sub = sub[sub[dcol] <= depth_max]
            if sub.empty:
                continue
            xlo, xhi = min(xlo, sub[xcol].min()), max(xhi, sub[xcol].max())
            ylo, yhi = min(ylo, sub[ycol].min()), max(yhi, sub[ycol].max())
            traces.append(go.Scatter(
                x=sub[xcol], y=sub[ycol], name=st,
                mode="lines+markers" if SHOW_MARKERS else "lines",
                line=dict(shape=LINE_SHAPE, width=LINE_WIDTH,
                          color=STATION_COLORS.get(st, station_color(i))),
                marker=dict(size=4),
                hovertemplate=(f"<b>{st}</b><br>{s['label']}: %{{x:.3f}}"
                               f"<br>{ylabel}: %{{y:.3f}}<extra></extra>"),
            ))
        if not traces:
            continue

        x0, x1 = _padded(xlo, xhi, X_PAD_FRAC)
        if is_depth:
            # Anchor the depth axis to the window that was ASKED for, not to the
            # outermost reading. Bins sit at bin centres (10.907 ... 19.831), so a
            # 10–20 m window holds no reading at exactly 10 or 20; anchoring to the
            # data would push both those lines off the frame. Blank means surface
            # to deepest, which keeps 0 m visible for the same reason.
            top_req = depth_min if depth_min is not None else 0.0
            bot_req = depth_max if depth_max is not None else yhi
            pad = (bot_req - top_req) * Y_PAD_FRAC or 1.0
            lo, hi = top_req - pad, bot_req + pad
        else:
            lo, hi = _padded(ylo, yhi, Y_PAD_FRAC)
        yrange = [hi, lo] if invert else [lo, hi]   # descending → down is deeper

        head = f"{y_channel['name']} vs {s['name']}"
        if LOCATION:
            head = f"{LOCATION}: {head}"

        # Inverted, the profile is read downward from the surface, so the x axis
        # belongs at the top where the reader starts. Upright, it is an ordinary
        # graph and the x axis goes back to the bottom. Margins follow the axis.
        x_side = "top" if invert else "bottom"
        top_m, bottom_m = (110, 40) if invert else (70, 70)

        fig = go.Figure(traces)
        fig.update_layout(
            title=dict(text=head, x=0.5, xanchor="center", font=dict(size=16)),
            xaxis=dict(range=[x0, x1], side=x_side,
                       title=dict(text=s["label"], font=dict(size=AXIS_FONT), standoff=8)),
            yaxis=dict(range=yrange,
                       title=dict(text=ylabel, font=dict(size=AXIS_FONT), standoff=8)),
            template="plotly_white", hovermode="closest",
            width=760, height=620,
            legend=dict(title="Station"),
            margin=dict(l=70, r=30, t=top_m, b=bottom_m),
        )
        figs.append((s["name"], fig))
    return figs


_HEMISPHERE = {"N": 1, "S": -1, "E": 1, "W": -1}


def parse_coordinate(text, kind="lat"):
    """Read a latitude or longitude written almost any way.

    Handles decimal degrees, degrees with decimal minutes, and degrees minutes
    seconds; degree, minute and second marks in ASCII or Unicode; a hemisphere
    letter anywhere in the string or a leading minus; comma decimal separators;
    and raw NMEA ddmm.mmm. Any number of decimal places.

        47.4012          47 24.072 N        47°24'04.32"N      -122 31 51.6
        47,4012          N 47 24.072        122:31:51.6 W      4724.072

    Returns (decimal_degrees, description) on success, or (None, reason)."""
    if text is None:
        return None, "empty"
    s = str(text).strip().upper()
    if not s:
        return None, "empty"

    for mark in ("°", "′", "″", "’", "‘", "´",
                 "ʼ", "'", '"', ":", "_"):
        s = s.replace(mark, " ")

    letters = {c for c in s if c in "NSEW"}
    if len(letters) > 1:
        return None, "more than one hemisphere letter"
    hemi = letters.pop() if letters else None
    if hemi and kind == "lat" and hemi in "EW":
        return None, f"{hemi} is a longitude direction"
    if hemi and kind == "lon" and hemi in "NS":
        return None, f"{hemi} is a latitude direction"

    negative = bool(re.match(r"\s*-", s))
    s = re.sub(r"[NSEW+\-]", " ", s)
    s = re.sub(r"(?<=\d),(?=\d)", ".", s)      # 47,4012 -> 47.4012
    s = s.replace(",", " ")

    if re.search(r"[A-Z]", s):
        return None, "unexpected letters"
    parts = re.findall(r"\d+(?:\.\d+)?", s)
    if not parts:
        return None, "no numbers found"
    if len(parts) > 3:
        return None, "too many numbers"

    vals = [float(p) for p in parts]
    # Only the last component may be fractional: "47 24.072" is fine, "47.4.012"
    # is a typo rather than degrees-and-minutes.
    if len(vals) > 1 and any(v != int(v) for v in vals[:-1]):
        return None, "only the last number may have decimals"
    minutes = vals[1] if len(vals) > 1 else 0.0
    seconds = vals[2] if len(vals) > 2 else 0.0
    if minutes >= 60:
        return None, "minutes must be under 60"
    if seconds >= 60:
        return None, "seconds must be under 60"

    magnitude = vals[0] + minutes / 60.0 + seconds / 3600.0
    form = ("decimal degrees", "degrees + decimal minutes",
            "degrees minutes seconds")[len(vals) - 1]
    limit = 90.0 if kind == "lat" else 180.0

    # A bare number too large to be degrees may be raw NMEA, which packs degrees
    # and minutes together: 4724.072 means 47° 24.072'.
    if len(vals) == 1 and magnitude > limit:
        deg, mins = divmod(vals[0], 100.0)
        if mins < 60 and deg + mins / 60.0 <= limit:
            magnitude, form = deg + mins / 60.0, "NMEA ddmm.mmm"

    sign = _HEMISPHERE[hemi] if hemi else (-1 if negative else 1)
    value = sign * magnitude
    if abs(value) > limit:
        return None, f"outside ±{limit:g}°"
    return value, form


def format_coordinate(value, kind="lat"):
    """Decimal degrees plus the same position in degrees minutes seconds."""
    if value is None:
        return "—"
    hemi = ("N" if value >= 0 else "S") if kind == "lat" else ("E" if value >= 0 else "W")
    a = abs(value)
    d = int(a)
    m = int((a - d) * 60)
    sec = (a - d - m / 60.0) * 3600.0
    return f"{value:.6f}°  ({d}° {m}' {sec:.2f}\" {hemi})"


def station_position(df, meta):
    """Best available position for a cast, as (lat, lon) or (None, None).

    A Sea-Bird CTD has no GPS of its own. Position arrives from the ship's GPS
    through the deck unit's NMEA input, or is typed into Seasave before the
    cast — so an instrument run self-contained has none at all. Where the deck
    unit was set to append position to every scan there are latitude and
    longitude columns as well, and their median is a fair cast position."""
    lat, lon = meta.get("lat"), meta.get("lon")
    if lat is not None and lon is not None:
        return lat, lon
    la = _col_by_name(df, "latitude")
    lo = _col_by_name(df, "longitude")
    if la is not None and lo is not None:
        try:
            v1, v2 = float(df[la].median()), float(df[lo].median())
            if -90 <= v1 <= 90 and -180 <= v2 <= 180:
                return v1, v2
        except (TypeError, ValueError):
            pass
    return None, None


def station_map(positions=None, title=None, connect=False):
    """Station positions on an OpenStreetMap basemap, coloured to match the
    profile graphs. Returns None when nothing has coordinates.

    positions: {label: (lat, lon)}. Needs internet for the map tiles, so the
    offline export will show markers on a blank background."""
    pts = positions if positions is not None else {
        k: (m.get("lat"), m.get("lon")) for k, m in STATION_META.items()}
    pts = {k: v for k, v in pts.items()
           if v and v[0] is not None and v[1] is not None}
    if not pts:
        return None

    labels = list(pts)
    lats = [pts[k][0] for k in labels]
    lons = [pts[k][1] for k in labels]
    mid_lat = sum(lats) / len(lats)
    # Longitude degrees shrink with latitude, so compare spans in like units.
    span = max(max(lats) - min(lats),
               (max(lons) - min(lons)) * float(np.cos(np.radians(mid_lat))))
    zoom = next(z for lim, z in [(0.02, 12), (0.05, 11), (0.2, 10), (0.5, 9),
                                 (1, 8), (5, 6), (20, 4), (60, 3), (1e9, 1)]
                if span < lim)

    # plotly 6 renamed the Mapbox traces to Map and moved to MapLibre. Colab is
    # still on 5.x, which has only the old names, so support both.
    Trace = getattr(go, "Scattermap", None) or go.Scattermapbox
    map_key = "map" if hasattr(go, "Scattermap") else "mapbox"

    traces = []
    if connect and len(labels) > 1:
        traces.append(Trace(
            lat=lats, lon=lons, mode="lines", name="transect",
            line=dict(width=2, color="#555"), hoverinfo="skip", showlegend=False))
    for k in labels:
        la, lo = pts[k]
        traces.append(Trace(
            lat=[la], lon=[lo], mode="markers+text", name=k,
            marker=dict(size=13, color=STATION_COLORS.get(k, "#1f77b4")),
            text=[k], textposition="top right",
            textfont=dict(size=12),
            hovertemplate=f"<b>{k}</b><br>%{{lat:.4f}}, %{{lon:.4f}}<extra></extra>"))

    head = title or (f"{LOCATION}: stations" if LOCATION else "Stations")
    fig = go.Figure(traces)
    fig.update_layout(
        title=dict(text=head, x=0.5, xanchor="center", font=dict(size=16)),
        width=760, height=620, margin=dict(l=10, r=10, t=60, b=10),
        legend=dict(title="Station"),
        **{map_key: dict(style="open-street-map",
                         center=dict(lat=mid_lat, lon=sum(lons) / len(lons)),
                         zoom=zoom)},
    )
    return fig


def write_combined_html(figs, path, title="CTD Profiles", inline=False, show_heading=True):
    """All figures in one HTML file.

    inline=False links the plotting library from the web: ~30 KB, opens at once,
    emails and uploads without trouble, but needs a connection to draw.
    inline=True bakes the library in: several MB, works with no internet. Keep
    that one for presenting somewhere without wifi — a file that large is slow
    to open and easy to truncate in transit, so it is a poor default."""
    parts = [f.to_html(full_html=False,
                       include_plotlyjs=("inline" if inline else "cdn") if i == 0 else False)
             for i, (_, f) in enumerate(figs)]
    html = (
        '<!doctype html><html><head><meta charset="utf-8">'
        f"<title>{title}</title><style>"
        "body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif;margin:24px;"
        "background:#fff;color:#111}h1{font-size:20px;font-weight:600}"
        ".grid{display:flex;flex-wrap:wrap;gap:16px}"
        # never let a figure be squeezed to nothing by the flex container
        ".grid>div{flex:0 0 auto}</style></head><body>"
        + (f"<h1>{title}</h1>" if show_heading else "")
        + f"<div class=\"grid\">{''.join(parts)}</div></body></html>"
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


def export_figures(figs, tag="", want_png=True):
    """Write the interactive HTML (+ optional PNGs) and return a status string."""
    os.makedirs(os.path.join(OUT_DIR, "png"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, "single_graphs"), exist_ok=True)
    base = f"{LOCATION} — CTD Profiles" if LOCATION else "CTD Profiles"
    html_path = os.path.join(OUT_DIR, "CTD_profiles.html")
    offline_path = os.path.join(OUT_DIR, "CTD_profiles_offline.html")
    write_combined_html(figs, html_path, title=f"{base}{tag}", inline=False)
    write_combined_html(figs, offline_path, title=f"{base}{tag}", inline=True)

    # One file per variable, so a single graph can be shared or embedded on its
    # own without anyone having to edit HTML. No heading: the figure carries its
    # own title, and a bare graph embeds more cleanly in someone else's page.
    for name, fig in figs:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
        one = f"{LOCATION}: Depth vs {name}" if LOCATION else f"Depth vs {name}"
        write_combined_html([(name, fig)],
                            os.path.join(OUT_DIR, "single_graphs", f"{safe}.html"),
                            title=one, inline=False, show_heading=False)

    msg = (f"{html_path} ({os.path.getsize(html_path)/1e3:.0f} KB)"
           f" + offline copy ({os.path.getsize(offline_path)/1e6:.1f} MB)"
           f" + {len(figs)} single graphs")
    if want_png and EXPORT_PNG:
        try:
            for name, fig in figs:
                safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
                fig.write_image(os.path.join(OUT_DIR, "png", f"{safe}.png"), scale=3)
            msg += f" · {len(figs)} PNGs"
        except Exception as e:
            msg += f" · PNG skipped ({type(e).__name__})"
    return msg


print("Setup complete.  markers:", SHOW_MARKERS, "· x-padding:", f"{X_PAD_FRAC:.0%}")

In [ ]:
#@title 1 · Survey location and files
#@markdown **Survey location** — the name of the overall area you sampled.
#@markdown It becomes the title of every graph, for example
#@markdown `Quartermaster Harbor: Depth vs Temperature`. Leave it blank to get
#@markdown just `Depth vs Temperature`.
survey_location = "" #@param {type:"string"}
#@markdown ---
#@markdown **Keep the downcast only.** The instrument records on the way down and
#@markdown again on the way back up. An unprocessed file holds both, so the line
#@markdown retraces itself. Ticked, only the downward half is kept. Files that
#@markdown were already processed are left alone.
#@markdown
#@markdown This fixes the **shape, not the numbers** — see Raw casts in the
#@markdown [README](https://github.com/jimothy-dev/CTD_Grapher_v2#raw-casts).
downcast_only_raw = True #@param {type:"boolean"}
#@markdown ---
#@markdown Run this cell and pick your `.cnv` files when the upload button appears.
#@markdown Picked the wrong file? Run this cell again and re-upload — only the
#@markdown newest copy of each station is used.

import ipywidgets as widgets
from IPython.display import display

LOCATION = survey_location.strip()
raw = load_files()

stations = []
# Sort on the cleaned station name, never the filename — a ' (1)' left by a
# re-upload would otherwise reorder stations and shuffle every colour.
for fn in sorted(raw, key=lambda f: natkey(station_label(f))):
    try:
        df, proc = parse_cnv(raw[fn], fn)
    except ValueError as e:
        print(f"  SKIPPED  {e}")
        continue
    if df.empty:
        print(f"  SKIPPED  {fn}: no data rows after *END*")
        continue

    label = station_label(fn)
    dcol, dlab = find_channel(df, DEPTH_CANDIDATES)

    note = ""
    if "loopedit" not in proc:
        if downcast_only_raw:
            df, dropped = downcast_only(df, dcol)
            # Not "loopedit" — that is a Sea-Bird step this notebook does not run.
            proc = proc | {"downcast cut (this notebook)"}
            note = f"  ← raw cast, kept the downcast ({dropped:,} rows dropped)"
        else:
            note = "  ← raw cast, may double back on itself"

    # A tidy-looking profile should not imply the values were corrected.
    missing = [s for s in ("align", "celltm") if not any(s in p for p in proc)]
    if missing and "derive" in proc:
        note += ("\n      NOTE  this file's header shows no correction steps, so the "
                 "values may be off. See Raw casts in the README.")

    stations.append((label, df))
    STATION_META[label] = parse_header_meta(raw[fn])
    drange = f"{df[dcol].min():.1f}–{df[dcol].max():.1f}" if dcol else "no depth column"
    print(f"  {label:<28} {len(df):>6,} rows   {drange:<16} "
          f"processing: {', '.join(sorted(proc)) or 'none'}{note}")

if not stations:
    raise SystemExit("No readable .cnv files found.")

DEEPEST = max(df[find_channel(df, DEPTH_CANDIDATES)[0]].max() for _, df in stations)
assign_station_styles([lab for lab, _ in stations])

print(f"\n{len(stations)} station(s) loaded · deepest reading {DEEPEST:.1f} m")
print(f"Location: {LOCATION or '(none set)'}")
print("\nColour key — reuse these in any other chart of the same stations:")
for lab, col in STATION_COLORS.items():
    print(f"  {col}   {lab}")


# ─────────── choose what to plot ───────────
RECOGNISED, EXTRAS, Y_DEFAULT = available_channels(stations)


def _row(chan, ticked):
    box = widgets.Checkbox(value=ticked, description=chan["name"], indent=False,
                           layout=widgets.Layout(width="250px"))
    txt = widgets.Text(value=chan["label"], layout=widgets.Layout(width="270px"))
    return box, txt, chan


# The classic variables start ticked. Derived and engineering channels are
# recognised but unticked, and anything unrecognised is offered unticked too, so
# nothing is hidden and the default output stays the usual set of graphs.
CHANNEL_ROWS = ([_row(c, c.get("on", True)) for c in RECOGNISED]
                + [_row(c, False) for c in EXTRAS])

Y_CHOICES = {c["name"]: c for c in [Y_DEFAULT] + RECOGNISED + EXTRAS}
Y_PICK = widgets.Dropdown(options=list(Y_CHOICES), value=Y_DEFAULT["name"],
                          description="Y axis:", style={"description_width": "62px"},
                          layout=widgets.Layout(width="290px"))
Y_LABEL = widgets.Text(value=Y_DEFAULT["label"], description="Y label:",
                       style={"description_width": "62px"},
                       layout=widgets.Layout(width="290px"))
Y_INVERT = widgets.Checkbox(value=True, description="Invert Y axis (largest at the bottom)",
                            indent=False, layout=widgets.Layout(width="360px"))
Y_PICK.observe(lambda _c: setattr(Y_LABEL, "value", Y_CHOICES[Y_PICK.value]["label"]),
               names="value")


def selected_series():
    """Ticked channels, with whatever axis label sits in the box beside each."""
    return [{"name": ch["name"], "col": ch["col"], "label": txt.value}
            for box, txt, ch in CHANNEL_ROWS if box.value]


def selected_y():
    # is_depth only decides whether the axis is anchored to the requested window
    # so 0 m stays visible. The depth window itself filters readings and applies
    # whatever is on the axes. invert is purely the tick box.
    ch = Y_CHOICES[Y_PICK.value]
    return {"name": ch["name"], "col": ch["col"], "label": Y_LABEL.value,
            "is_depth": bool(ch.get("is_depth", False)),
            "invert": bool(Y_INVERT.value)}


display(widgets.HTML(
    "<b>Plot these</b> &mdash; tick what you want a graph of. The box beside each "
    "is its axis label, units included; edit it if you like.<br>"
    "Recognised variables are already ticked. The rest are other columns your "
    "files happen to contain, in case you want them."))
display(widgets.VBox([widgets.HBox([b, t]) for b, t, _ in CHANNEL_ROWS]))
display(widgets.HTML(
    "<b>Y axis</b> &mdash; depth unless you change it. Pick something else and the "
    "graphs become ordinary scatter plots titled <i>Salinity vs Temperature</i> "
    "and so on. The depth window in the next cell still applies."))
display(widgets.HBox([Y_PICK, Y_LABEL]))
display(Y_INVERT)


# ─────────── station positions ───────────
# A Sea-Bird CTD has no GPS. Position comes from the ship's GPS via the deck
# unit's NMEA input, or is typed into Seasave before the cast, so plenty of
# files have none. These boxes are prefilled where the file knows, and typed in
# where it does not.
def _pos_row(label, df, meta):
    lat, lon = station_position(df, meta)
    name = widgets.HTML(f"<div style='width:150px;padding-top:4px'>{label}</div>")
    la = widgets.Text(value="" if lat is None else f"{abs(lat):.5f}",
                      placeholder="latitude", layout=widgets.Layout(width="150px"))
    ns = widgets.Dropdown(options=["N", "S"], value="S" if (lat or 0) < 0 else "N",
                          layout=widgets.Layout(width="62px"))
    lo = widgets.Text(value="" if lon is None else f"{abs(lon):.5f}",
                      placeholder="longitude", layout=widgets.Layout(width="150px"))
    ew = widgets.Dropdown(options=["E", "W"], value="W" if (lon or 0) < 0 else "E",
                          layout=widgets.Layout(width="62px"))
    out = widgets.HTML(layout=widgets.Layout(width="430px"))
    row = (name, la, ns, lo, ew, out, label)

    def refresh(_change=None):
        # A hemisphere letter or minus typed into the box wins over the
        # dropdown, so pasted coordinates behave as written.
        v1, n1 = parse_coordinate(la.value, "lat")
        v2, n2 = parse_coordinate(lo.value, "lon")
        if v1 is not None and not re.search(r"[NSns\-]", la.value):
            v1 = abs(v1) * (-1 if ns.value == "S" else 1)
        if v2 is not None and not re.search(r"[EWew\-]", lo.value):
            v2 = abs(v2) * (-1 if ew.value == "W" else 1)
        bits = []
        for v, note, kind, raw in ((v1, n1, "lat", la.value), (v2, n2, "lon", lo.value)):
            if raw.strip() == "":
                continue
            bits.append(f"<span style='color:#0a0'>{format_coordinate(v, kind)}</span>"
                        if v is not None else
                        f"<span style='color:#b00'>{kind}: {note}</span>")
        out.value = ("<div style='padding-top:4px;font-family:monospace;font-size:11px'>"
                     + " &nbsp; ".join(bits) + "</div>")
        m = STATION_META.setdefault(label, {})
        m["lat"], m["lon"] = v1, v2

    for w in (la, lo, ns, ew):
        w.observe(refresh, names="value")
    refresh()
    return row


POSITION_ROWS = [_pos_row(lab, df, STATION_META.get(lab, {})) for lab, df in stations]


def apply_positions():
    """STATION_META is kept current by the boxes themselves; this just reports."""
    return {k: (m.get("lat"), m.get("lon")) for k, m in STATION_META.items()}


_have = sum(1 for r in POSITION_ROWS if r[1].value)
display(widgets.HTML(
    f"<b>Station positions</b> &mdash; {_have} of {len(POSITION_ROWS)} came from the "
    f"file headers. A Sea-Bird CTD has no GPS of its own, so a cast run without a "
    f"deck unit feeding it NMEA records no position; type those in for a station map."
    f"<br>Any usual format works &mdash; <code>47.4012</code>, "
    f"<code>47 24.072</code>, <code>47°24'04.3\"</code>, <code>4724.072</code> "
    f"&mdash; and the converted value is shown beside each box."))
display(widgets.VBox([widgets.HBox([n, la, ns, lo, ew, out])
                      for n, la, ns, lo, ew, out, _lab in POSITION_ROWS]))
print("\nNow run the Plot cell.")

In [ ]:
#@title 2 · Draw the graphs
#@markdown **Which part of the water column to show**, in metres below the surface.
#@markdown Leave both blank to show the whole cast, top to bottom.
#@markdown
#@markdown To look at just the surface layer, put `0` and `20` — that shows
#@markdown everything between 0 and 20 metres deep. Change the numbers and run
#@markdown this cell again; your files stay loaded. Applies whatever is on the
#@markdown axes, since it filters readings by how deep they were taken.
depth_from_m = "" #@param {type:"string"}
depth_to_m   = "" #@param {type:"string"}

try:
    stations
except NameError:
    raise SystemExit("Run the Load cell first.")


def _num(s):
    s = str(s).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        print(f"  ignoring '{s}' — that is not a number")
        return None


y = selected_y()
series = [s for s in selected_series() if s["col"].lower() != y["col"].lower()]

if not series:
    raise SystemExit("Nothing ticked to plot. Tick at least one variable in the Load cell.")

dmin, dmax = _num(depth_from_m), _num(depth_to_m)
if dmin is not None and dmax is not None and dmin > dmax:
    dmin, dmax = dmax, dmin

figs = build_figures(stations, series, y, dmin, dmax)

if not figs:
    where = f" between {dmin:g} and {dmax:g} m" if (dmin or dmax) else ""
    print(f"Nothing to draw{where}. The deepest reading in this set is {DEEPEST:.1f} m.")
else:
    span = ""
    if dmin is not None or dmax is not None:
        lo = f"{dmin:g}" if dmin is not None else "0"
        hi = f"{dmax:g}" if dmax is not None else f"{DEEPEST:.0f}"
        span = f" · {lo}–{hi} m"
    # Station map goes first when any position is known, from the file or typed in.
    apply_positions()
    mp = station_map()
    if mp is not None:
        figs = [("Station map", mp)] + figs
        print(f"Station map: {sum(1 for m in STATION_META.values() if m.get('lat') is not None)}"
              f" of {len(stations)} stations positioned")
    else:
        print("Station map: skipped, no positions entered")

    print(f"{y['name']} vs: " + ", ".join(n for n, _ in figs if n != "Station map") + span)
    print("Saved:", export_figures(figs, tag=span))
    # Plain fig.show() with no clear_output(). Colab's plotly renderer attaches a
    # MutationObserver that purges the plot when its output element is rebuilt, so
    # clearing and redrawing the cell destroys the figures as they arrive.
    for _, f in figs:
        f.show()